# BT4012 final models — e1303010
**Wang TianAi · A0288096L**  
Repository: https://github.com/WangTianai920/BT4012-Kaggle

This notebook trains **both selected submissions, 1019-6 and 0929-1**, from the four raw competition files. It exposes preprocessing, feature engineering, the final training recipes, prediction checks and CSV export. The supporting modules in `src/` and `final/` are required. Run all cells in order from this repository root using Python 3.10 after installing `requirements.txt`.

## 1 Setup and inputs
Place `train.csv`, `test.csv`, `txs_edgelist.csv` and `sample_submission.csv` in `data/`. Alternatively set `BT4012_DATA_DIR` to their directory. Optional `BT4012_REFERENCE_DIR` points to the original submissions.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time
from types import SimpleNamespace
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'
import numpy as np
import pandas as pd
import joblib
from scipy.special import expit, logit
from IPython.display import display
ROOT = Path.cwd().resolve()
assert (ROOT / 'run_models.py').is_file(), 'Open this notebook from the repository root.'
DATA_DIR = Path(os.environ.get('BT4012_DATA_DIR', ROOT / 'data')).resolve()
REFERENCE_DIR = os.environ.get('BT4012_REFERENCE_DIR')
a = SimpleNamespace(data_dir=DATA_DIR, work_dir=ROOT/'work', output_dir=ROOT/'outputs',
                    reference_dir=Path(REFERENCE_DIR) if REFERENCE_DIR else None)
a.work_dir.mkdir(exist_ok=True); a.output_dir.mkdir(exist_ok=True)
required = ['train.csv','test.csv','txs_edgelist.csv','sample_submission.csv']
assert all((DATA_DIR / name).is_file() for name in required), 'Missing raw competition CSV files.'
t0 = time.time()
results = {}
receipt = {'raw_sha256': {name: hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in required}}
import src.competition as competition
competition.DATA_DIR = DATA_DIR
train_raw, test_raw, edges_raw, sample = competition.load_competition_data()
print('Four input files loaded; hashes recorded for reproducibility.')

Four input files loaded; hashes recorded for reproducibility.


## 2 Data preprocessing and integrity checks
0929-1 excludes unknown labels from supervised fitting, while 1019-6 models them as a third observation status. Check identifiers, temporal boundaries and numerical inputs before building features. No imputation is needed for the supplied complete attributes.

In [ ]:
raw_columns = competition.feature_columns(train_raw)
assert len(raw_columns) == 165
assert train_raw.txId.is_unique and test_raw.txId.is_unique
assert not set(train_raw.txId).intersection(test_raw.txId)
assert train_raw.time_step.between(1,35).all()
assert test_raw.time_step.between(36,49).all()
assert set(train_raw.label.dropna().unique()) == {0,1}
assert np.isfinite(train_raw[raw_columns].to_numpy()).all()
assert np.isfinite(test_raw[raw_columns].to_numpy()).all()
assert not edges_raw.duplicated().any()
assert not (edges_raw.txId1 == edges_raw.txId2).any()
assert sample['index'].equals(test_raw['index'])
display(pd.DataFrame({'count': {
    'Training rows': len(train_raw), 'Known licit': int((train_raw.label==0).sum()),
    'Known illicit': int((train_raw.label==1).sum()), 'Unknown labels': int(train_raw.label.isna().sum()),
    'Test rows': len(test_raw), 'Directed edges': len(edges_raw)}}))

,count
Directed edges,234355
Known illicit,3644
Known licit,27591
Test rows,15329
Training rows,141772
Unknown labels,110537


## 3 Feature engineering
The selected inputs use label-free structural features; some auxiliary caches are built for legacy module compatibility but are not selected predictors. Both recipes remove the ten raw attributes with the largest normalised range of period medians in steps 1–14. The binary model retains 172 inputs; the hybrid adds nine chain-position, fragmentation and block features for 181 inputs. The GNN worker fits its scaler on history, clips standardised values to ±5, and uses missing-attribute masks/tokens and 25% attribute-row masking.

In [ ]:
from final.regenerate_features import regenerate
from final import pipeline
regenerate(a.work_dir)
inp = pipeline.load_inputs(a.work_dir)
train, test, sample = inp['train'], inp['test'], inp['sample']
from src.horizon_hgb_20260917 import historical_columns
from src.combined_challengers import load_features
tr, te, _, groups = load_features()
binary_columns = historical_columns(tr, groups['regularized_two_hop'], 35)
history, hybrid_columns, statuses, weights = pipeline.history_view(inp, 35)
dropped = [name for name in raw_columns if name not in binary_columns]
assert len(binary_columns)==172 and len(hybrid_columns)==181 and len(dropped)==10
node_period = inp['node_map'].set_index('txId')['period']
assert edges_raw.txId1.map(node_period).equals(edges_raw.txId2.map(node_period))
print('Binary inputs:', len(binary_columns), '| Hybrid inputs:', len(hybrid_columns))
print('Removed raw features:', ', '.join(dropped))

[regen] tabular-derived caches (graph, graph-structure, within-step ranks)


[regen] GRAPH5 directed chain-position features (all 203,769 graph nodes)


[regen] terminal-coverage node map (period-pure weak components) + deduplicated edges


[regen] CUT2 articulation-cut features


[regen] BLOCK2 biconnected-block features


Binary inputs: 172 | Hybrid inputs: 181
Removed raw features: feat_100, feat_101, feat_103, feat_113, feat_134, feat_135, feat_136, feat_139, feat_149, feat_151


## 4 Train 0929-1
Fit a binary HGB base model using known labels through period 35 and a 20-period recency half-life. Historical specialist training features for periods 29–35 come from a first-stage model fitted through period 28. Blend the base, original/enhanced specialists and five stratified bootstrap members, then apply 0.10 median-logit alignment. This changes cross-period score levels but preserves within-period ranks.

In [ ]:
print('Rebuilding 0929-1 binary HGB and historical risk specialists',flush=True)
from src.combined_challengers import load_features
from src.horizon_hgb_20260917 import predict_stack,historical_columns,BASE
from src.score_level_alignment_20260929 import align
_,_,edges,_=competition.load_competition_data()
tr,te,_,groups=load_features()
cols=historical_columns(tr,groups['regularized_two_hop'],35)
stack,diag=predict_stack(tr,te,edges,groups['regularized_two_hop'],cols,35,BASE)
score,med,ref=align(stack['bagged'],te.time_step,.10)
results['0929-1']=score
receipt['0929-1']={'features':len(cols),'diagnostics':diag,'period_medians':med,'reference_median':ref}

Rebuilding 0929-1 binary HGB and historical risk specialists


## 5 Train the 1019-6 teacher
Build separate historical and forecast graph partitions. Use three statuses and 181 inputs. Apply a 20-period recency half-life, an extra three-period half-life fade before period 18, and rescale unknown status to 40% of total training weight. Fit the HGB teacher for 600 boosting iterations. Tree fitting occurs before any Torch worker is started to avoid mixed-runtime issues.

In [ ]:
print('Building 1019-6 graph partitions',flush=True)
h,cols,status,w=pipeline.history_view(inp,35)
future=test.copy();future['label']=np.nan
hp=pipeline.partition(h,range(1,36),inp['node_map'],inp['edges'],inp['graph9'],cols,status,w)
fp=pipeline.partition(future,range(36,50),inp['node_map'],inp['edges'],inp['graph9'],cols)
# Fit the tree before importing Torch; mixed OpenMP runtimes can stall on macOS.
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits
threadpool_limits(1)
soft_weights=hp['weights'].astype(float).copy()
old=hp['periods']<18
soft_weights[old]*=0.5**((18-hp['periods'][old])/3.0)
unknown=hp['labels']==2;known=(hp['labels']==0)|(hp['labels']==1)
soft_weights[unknown]*=(.4/.6)*soft_weights[known].sum()/max(soft_weights[unknown].sum(),1e-12)
hp['weights']=soft_weights
attr=hp['is_attr'];selected=(hp['labels'][attr]>=0)&(soft_weights[attr]>0)
Xh=pd.DataFrame(hp['raw'][attr],columns=cols)
print('Fitting soft-window HGB teacher in the tree-only process',flush=True)
teacher=HistGradientBoostingClassifier(l2_regularization=2.0,learning_rate=.03,max_features=.8,max_iter=600,max_leaf_nodes=15,min_samples_leaf=40,early_stopping=False,random_state=42)
teacher.fit(Xh[selected],hp['labels'][attr][selected],sample_weight=soft_weights[attr][selected])
for part in (hp,fp):
    part['teacher_logits']=np.zeros((len(part['ids']),3),dtype=np.float32)
    probabilities=teacher.predict_proba(pd.DataFrame(part['raw'][part['is_attr']],columns=cols))
    part['teacher_logits'][part['is_attr']]=np.log(np.clip(probabilities,1e-12,1)).astype(np.float32)
print('Soft-window teacher completed',flush=True)
job=dict(history=hp,future=fp,columns=cols,graph_columns=pipeline.GRAPH9,origin=35,stop=49,stage='final',configuration=pipeline.gnn_config(),extra={'min_period':18,'soft_hl':3.0})
ip=a.work_dir/'softwindow_inputs.joblib';joblib.dump(job,ip,compress=3)

Building 1019-6 graph partitions


Fitting soft-window HGB teacher in the tree-only process


Soft-window teacher completed


['/private/tmp/bt4012_notebook_build/repo/work/softwindow_inputs.joblib']

## 6 Train the graph correction and ensemble
A directed gated-mean graph network learns tanh-bounded class-logit corrections to the frozen teacher. It uses width 32, 60 epochs, AdamW (learning rate/weight decay 0.001), dropout 0.1 and 25% attribute-row masking. Class probabilities are converted to illicit/(licit+illicit). Five random seeds are averaged in log-odds space; no period adjustment is applied to this model. The subprocesses use this notebook kernel's Python interpreter.

In [ ]:
logits=[]
for seed in [42,101,202,303,404]:
    print(f'Fitting 1019-6 seed {seed}',flush=True)
    op=a.work_dir/f'softwindow_seed{seed}.joblib'
    code="import sys,joblib;import src.lite.worker_gm_tunable_20261014 as G;G.SEED=int(sys.argv[3]);joblib.dump(G.fit(joblib.load(sys.argv[1])),sys.argv[2])"
    subprocess.run([sys.executable,'-c',code,str(ip),str(op),str(seed)],cwd=ROOT,check=True)
    result=joblib.load(op);sc=np.asarray(result['score'],float)[fp['attributed_positions']]
    logits.append(logit(np.clip(sc,1e-12,1-1e-12)))
results['1019-6']=expit(np.mean(logits,axis=0))
receipt['1019-6']={'seeds':[42,101,202,303,404],'features':len(cols),'min_period':18,'soft_half_life':3,'history_status_counts':np.bincount(status,minlength=3).tolist()}

Fitting 1019-6 seed 42


Fitting 1019-6 seed 101


Fitting 1019-6 seed 202


Fitting 1019-6 seed 303


Fitting 1019-6 seed 404


## 7 Validate predictions and generate both submission CSVs
Check probability range, finiteness, nonconstant predictions, unique IDs, column names and template order. These are output-integrity checks, not estimates of test AUC. If references are supplied, compare generated predictions to the original files with tolerance 1e-5. Write `outputs/1019-6.csv`, `outputs/0929-1.csv` and a machine-readable receipt.

In [ ]:
failures=[]

for name,score in results.items():
    assert len(score)==len(sample)==15329 and np.isfinite(score).all() and ((score>=0)&(score<=1)).all() and np.std(score)>0
    frame=pd.DataFrame({'index':sample['index'],'target':score});path=a.output_dir/f'{name}.csv';frame.to_csv(path,index=False)
    item=receipt[name];item['output_sha256']=hashlib.sha256(path.read_bytes()).hexdigest();item['rows']=len(frame)
    if a.reference_dir:
        matches=[p for p in a.reference_dir.glob('*.csv') if ''.join(p.stem.split())==name]
        if len(matches)!=1:raise ValueError(f'Expected one reference file for {name}; found {len(matches)}')
        reference=pd.read_csv(matches[0])
        assert np.array_equal(reference['index'],frame['index'])
        delta=float(np.max(np.abs(reference.target-pd.read_csv(path).target)))
        item['max_absolute_difference']=delta;item['acceptance_tolerance']=1e-5;item['accepted']=delta<=1e-5
        print(name,'max absolute difference:',delta,'PASS' if item['accepted'] else 'FAIL',flush=True)
        if not item['accepted']:failures.append(name)
receipt['runtime_seconds']=time.time()-t0
(a.output_dir/'reproduction_receipt.json').write_text(json.dumps(receipt,indent=2))
if failures:raise RuntimeError('Reproduction failed: '+', '.join(failures))
print('Completed in',round(time.time()-t0),'seconds',flush=True)

0929-1 max absolute difference: 0.0 PASS


1019-6 max absolute difference: 0.0 PASS


Completed in 447 seconds


In [ ]:
import numpy as np
import pandas as pd

def validate_submission(frame, sample):
    assert list(frame.columns) == ['index', 'target']
    assert 'index' in sample.columns
    assert len(frame) == len(sample)
    assert frame['index'].is_unique
    assert np.array_equal(frame['index'].to_numpy(), sample['index'].to_numpy())
    assert pd.api.types.is_numeric_dtype(frame['target'])
    t = frame['target'].to_numpy(dtype=float)
    assert np.isfinite(t).all()
    assert ((t >= 0) & (t <= 1)).all()
    assert np.unique(t).size > 1
    return {'checks_passed': True, 'row_count': int(len(frame))}

summary = []
for name in ['1019-6','0929-1']:
    frame = pd.read_csv(a.output_dir / f'{name}.csv')
    validate_submission(frame, sample)
    assert frame.columns.tolist() == ['index','target']
    assert frame['index'].is_unique
    assert np.array_equal(frame['index'], sample['index'])
    assert len(frame) == len(sample) == 15329
    assert np.isfinite(frame.target).all() and frame.target.between(0,1).all()
    assert frame.target.nunique() > 1
    summary.append({'submission':name, 'rows':len(frame), 'minimum':frame.target.min(),
                    'maximum':frame.target.max(), 'integrity':'PASS',
                    'reference_max_difference':receipt[name].get('max_absolute_difference', np.nan)})
display(pd.DataFrame(summary))
print('Generated outputs/1019-6.csv and outputs/0929-1.csv.')

,submission,rows,minimum,maximum,integrity,reference_max_difference
0,1019-6,15329,5.094551e-07,0.999915,PASS,0.0
1,0929-1,15329,1.492605e-05,0.999540,PASS,0.0


Generated outputs/1019-6.csv and outputs/0929-1.csv. No upload performed.
